# ONIA · 2.7 Mașini cu vectori suport (SVM)

Exemple pentru elevi: geometrie, hinge loss, clasificare binară/multiclasă, kerneluri, evaluare și SVR.

## Geometria unei drepte și clasificarea

In [ ]:
def scor(x, w, b):
    return sum(x[i] * w[i] for i in range(len(x))) + b

for x in [[3, 1], [1, 3]]:
    print(x, scor(x, [2, -1], -3))

## Hinge loss

In [ ]:
def hinge(y, f):
    return max(0, 1 - y * f)
print([hinge(y, f) for y, f in [(1, 2), (1, .4), (-1, .5), (-1, -2)]])

## Clasificare liniară de la zero

In [ ]:
def antreneaza(X, y, epoci=300, pas=.02, C=2):
    w = [0.0] * len(X[0]); b = 0.0
    for _ in range(epoci):
        dw = w.copy(); db = 0.0
        for x, et in zip(X, y):
            if et * (sum(w[j]*x[j] for j in range(len(w))) + b) < 1:
                for j in range(len(w)):
                    dw[j] -= C * et * x[j] / len(X)
                db -= C * et / len(X)
        for j in range(len(w)):
            w[j] -= pas * dw[j]
        b -= pas * db
    return w, b
X = [[-2,-1], [-1,-2], [1,2], [2,1]]
y = [-1,-1,1,1]
print(antreneaza(X,y))

## Un SVC pe date sintetice

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
X, y = make_moons(n_samples=220, noise=.18, random_state=4)
Xt, Xv, yt, yv = train_test_split(X, y, random_state=4, stratify=y)
model = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=3, gamma='scale'))
model.fit(Xt, yt)
print('Accuracy test:', accuracy_score(yv, model.predict(Xv)))

## Comparație linear vs RBF

In [ ]:
from sklearn.svm import SVC
for kernel in ['linear','rbf']:
    model = make_pipeline(StandardScaler(), SVC(kernel=kernel,C=2))
    model.fit(Xt,yt)
    print(kernel, round(accuracy_score(yv,model.predict(Xv)),3))

## Vectori suport

In [ ]:
model = SVC(kernel='linear').fit(Xt,yt)
print('Număr vectori suport:', len(model.support_))
print('Primii indici:', model.support_[:10])

## Clasificare Iris cu trei clase

In [ ]:
from sklearn.datasets import load_iris
from sklearn.svm import SVC, LinearSVC
Xi, yi = load_iris(return_X_y=True)
for est in [SVC(kernel='rbf'),LinearSVC(max_iter=10000)]:
    model = make_pipeline(StandardScaler(),est).fit(Xi, yi)
    print(type(est).__name__,model.predict([Xi[0],Xi[70],Xi[140]]))

## OvO și OvR: numărul de decizii

In [ ]:
for k in [3,4,10]:
    print(k, 'clase: OvO=',k*(k-1)//2,'OvR=',k)

## Reglarea C și gamma fără data leakage

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
pipe = Pipeline([('scaler',StandardScaler()), ('svc', SVC())])
grid = GridSearchCV(pipe, {'svc__C':[.1,1,10], 'svc__gamma':[.01,.1,1]}, cv=3, scoring='f1')
grid.fit(Xt,yt)
print(grid.best_params_)
print('F1 test:',grid.score(Xv,yv))

## Evaluare prin matricea de confuzie

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
best = grid.best_estimator_
print(confusion_matrix(yv,best.predict(Xv)))
print(classification_report(yv,best.predict(Xv)))

## SVR pentru o țintă numerică

In [ ]:
import numpy as np
from sklearn.svm import SVR
xr = np.arange(1,9).reshape(-1,1)
yr = [1.1,1.8,3.2,3.1,4.7,5.0,6.1,6.8]
reg = make_pipeline(StandardScaler(),SVR(kernel='rbf',C=10,epsilon=.2))
reg.fit(xr,yr)
print(reg.predict([[4.5]]))